# Follower pressure

A pressure that stays normal to the deformed shell adds to the residual of
a nonlinear analysis a force vector that depends on the displacements,
$\boldsymbol{F}_p(\boldsymbol{u})$ of `update_fext_follower`, and to its
Jacobian the load stiffness matrix
$\boldsymbol{K}_F = -\partial \boldsymbol{F}_p/\partial \boldsymbol{u}$ of
`update_KCfollower`, unsymmetric in general [Hibbitt1979],
[SchweizerhofRamm1984], see [Follower pressure](../follower_pressure.rst):

$$
\boldsymbol{R}(\boldsymbol{u}) = \boldsymbol{F}_{int}(\boldsymbol{u}) -
\boldsymbol{F}_p(\boldsymbol{u}), \qquad
\boldsymbol{K}_T = \boldsymbol{K}_{C0} + \boldsymbol{K}_{CNL} +
\boldsymbol{K}_{GNL} + \boldsymbol{K}_G + \boldsymbol{K}_F
$$

This example verifies $\boldsymbol{K}_T$ of one element with a Taylor test,
and computes the buckling pressure of a ring under a follower and under a
dead pressure, against the closed-form solution. The helper functions are
those of the tests `tests/test_follower_pressure.py` and
`tests/test_follower_pressure_validation.py`, imported from there; the
same checks are made by these tests.

In [1]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.join('..', 'tests'))
from test_follower_pressure import (KINEMATICS, TRUNCATIONS,
                                    residual_and_tangent, random_state)
from test_follower_pressure_validation import (ring_buckling_pressure,
                                               ring_closed_form)

from IPython.display import Markdown


def table(rows, fmt='{:.5g}'):
    """Markdown table of a list of dicts with the same keys"""
    keys = list(rows[0])
    cell = lambda v: fmt.format(v) if isinstance(v, float) else str(v)
    lines = ['| ' + ' | '.join(keys) + ' |', '|' + '---|'*len(keys)]
    lines += ['| ' + ' | '.join(cell(r[k]) for k in keys) + ' |' for r in rows]
    return Markdown('\n'.join(lines))

## Taylor test of the tangent stiffness matrix

One element of a laminated cylinder, $R = 203.2$ mm, with the pressure
$p = -2 \cdot 10^5 (1 + 0.3 \xi - 0.2 \eta^2 + 0.1 \xi \eta)$ Pa at the
integration points, multiplied by $10^3$ so that $\boldsymbol{K}_F$ is a
sizable part of $\boldsymbol{K}_T$. `residual_and_tangent` returns the
functions $\boldsymbol{R}(\boldsymbol{u})$, $\boldsymbol{K}_T(\boldsymbol{u})$
and $\boldsymbol{K}_F(\boldsymbol{u})$ of the element, for either
kinematics and either truncation of the area vector: `quadratic=0`
($\boldsymbol{a}_0 + \boldsymbol{a}_1$) or `quadratic=1`
($\boldsymbol{a}_0 + \boldsymbol{a}_1 + \boldsymbol{a}_2$).

For a state $\boldsymbol{u}$, a direction $\boldsymbol{d}$ and a step $h$,
the relative remainder

$$
e(h) = \frac{\| \boldsymbol{R}(\boldsymbol{u} + h \boldsymbol{d}) -
\boldsymbol{R}(\boldsymbol{u}) - h \boldsymbol{K}_T(\boldsymbol{u})
\boldsymbol{d} \|}{\| h \boldsymbol{K}_T(\boldsymbol{u}) \boldsymbol{d} \|}
$$

falls in proportion to $h$ when $\boldsymbol{K}_T$ is the Jacobian of
$\boldsymbol{R}$, the remainder of the Taylor expansion being of order
$h^2$, and stays at a plateau when it is not. $\boldsymbol{u}$ and
$\boldsymbol{d}$ are random states of the order of the thickness.

In [2]:
steps = np.array([1e-2, 1e-3, 1e-4, 1e-5])
taylor = {}
for kinematics in sorted(KINEMATICS):
    for quadratic in TRUNCATIONS:
        residual, tangent, follower, prop = residual_and_tangent(kinematics,
                                                                 quadratic)
        u = random_state(prop, 0)
        d = random_state(prop, 10)
        KTd = tangent(u) @ d
        r0 = residual(u)
        e = [np.linalg.norm(residual(u + h*d) - r0 - h*KTd)/np.linalg.norm(h*KTd)
             for h in steps]
        share = np.linalg.norm(follower(u) @ d)/np.linalg.norm(KTd)
        taylor[(kinematics, quadratic)] = np.array(e), share

fig, ax = plt.subplots(figsize=(5, 3.8))
for (kinematics, quadratic), (e, share) in taylor.items():
    ax.loglog(steps, e, 'o-', label=f'{kinematics}, quadratic={quadratic}')
ax.loglog(steps, steps*taylor[('donnell', 0)][0][0]/steps[0], 'k--', lw=0.8,
          label='slope 1')
ax.set_xlabel('$h$')
ax.set_ylabel('$e(h)$')
ax.legend(fontsize=8)
plt.show()

The order of $e(h)$, the slope $\log_{10}(e(h_i)/e(h_{i+1}))$ between
successive decades, against the order 1 of a consistent Jacobian, and the
share $\| \boldsymbol{K}_F \boldsymbol{d} \| / \| \boldsymbol{K}_T
\boldsymbol{d} \|$ of the load stiffness:

In [3]:
rows = []
for (kinematics, quadratic), (e, share) in taylor.items():
    order = np.log10(e[:-1]/e[1:]).mean()
    rows.append({'kinematics': kinematics, 'quadratic': quadratic,
                 'e(1e-2)': e[0], 'e(1e-5)': e[-1], 'order': order,
                 'reference order': 1., 'rel. diff. (%)': 100*(order - 1.),
                 'share of K_F': share})
table(rows)

| kinematics | quadratic | e(1e-2) | e(1e-5) | order | reference order | rel. diff. (%) | share of K_F |
|---|---|---|---|---|---|---|---|
| donnell | 0 | 0.00035292 | 3.5292e-07 | 1 | 1 | 0.00012076 | 0.053957 |
| donnell | 1 | 0.00035378 | 3.5377e-07 | 1 | 1 | 0.00056633 | 0.05421 |
| sanders | 0 | 0.00035985 | 3.5982e-07 | 1 | 1 | 0.0011758 | 0.054783 |
| sanders | 1 | 0.00036028 | 3.6024e-07 | 1 | 1 | 0.0017411 | 0.054996 |

## Buckling of a ring under external pressure

A long cylinder under uniform external pressure buckles as a ring. The
ring, $R = 1$ m and $h = 10$ mm, isotropic, is modelled by
`ring_buckling_pressure` with one element along $x$ and 192 around the
closed circumference. At every node $u$, $u_{,x}$, $u_{,y}$, $v_{,x}$,
$w_{,x}$ and $w_{,xy}$ are constrained, the generalized plane strain of a
ring, and the symmetry planes $y = 0, \pi R/2, \pi R, 3 \pi R/2$ have
$v = w_{,y} = 0$, which removes the rigid-body motions and keeps the modes
with an even number $n$ of circumferential waves. $\boldsymbol{K}_G$ is that
of the linear static solution of a unit external pressure, and the linear
buckling problem

$$
\left( \boldsymbol{K}_{C0} + \lambda (\boldsymbol{K}_G +
\boldsymbol{K}_{F}) \right) \boldsymbol{\phi} = \boldsymbol{0}
$$

of the follower pressure, or without $\boldsymbol{K}_{F}$ for the dead
pressure, is solved with `structsolve.lb`; $\lambda$ is the buckling
pressure.

**Reference.** `ring_closed_form` is the exact buckling pressure of a ring
with the same strains and load terms as the element, for the mode
$w = W \cos(n y/R)$, $v = V \sin(n y/R)$ and the prestress
$N_{yy} = p R$. With $\varepsilon_{yy} = v_{,y} + w/R$,
$\kappa_{yy} = -w_{,yy} + k_v v_{,y}$ and $\beta_y = -w_{,y} + k_v v$, with
$k_v = 0$ (Donnell-type) or $1/R$ (Sanders-type), the second variation of
the total potential is

$$
U_2 = \frac{1}{2} \int \left( A_{22} \varepsilon_{yy}^2 + D_{22}
\kappa_{yy}^2 + N_{yy} \beta_y^2 \right) dy - \frac{1}{2} \int p
\left( v \, (-w_{,y} + k_v v) + w \, (v_{,y} + w/R) \right) dy
$$

the last integral, the work of the first-order area vector, being left out
for the dead pressure; the pressure that makes $U_2$ singular for $n = 2$
is the reference. In the inextensional limit, $\varepsilon_{yy} = 0$,
$\lambda R^3/D_{22}$ is 3 (Sanders-type, follower), 4 (dead) and 16/5
(Donnell-type, follower); for
$R/h = 100$ the extensibility changes these by a relative amount of order
$(h/R)^2$.

In [4]:
inextensional = {('sanders', 'follower'): 3., ('sanders', 'dead'): 4.,
                 ('donnell', 'follower'): 16/5, ('donnell', 'dead'): 4.}
R = 1.
rows = []
for kinematics in sorted(KINEMATICS):
    for load in ['follower', 'dead']:
        lam, prop = ring_buckling_pressure(kinematics, load, R=R)
        A22, D22 = prop.A[1, 1], prop.D[1, 1]
        ref = -ring_closed_form(kinematics, load, 2, A22, D22, R)
        rows.append({'kinematics': kinematics, 'pressure': load,
                     'lambda R^3/D22': lam*R**3/D22,
                     'closed form': ref*R**3/D22,
                     'rel. diff. (%)': 100*(lam - ref)/ref,
                     'inextensional': inextensional[(kinematics, load)],
                     'rel. diff. inext. (%)':
                         100*(lam*R**3/D22/inextensional[(kinematics, load)] - 1)})
table(rows)

| kinematics | pressure | lambda R^3/D22 | closed form | rel. diff. (%) | inextensional | rel. diff. inext. (%) |
|---|---|---|---|---|---|---|
| donnell | follower | 3.2 | 3.2 | 4.4881e-06 | 3.2 | -0.00052884 |
| donnell | dead | 4 | 4 | 4.3479e-06 | 4 | 4.3478e-06 |
| sanders | follower | 3 | 3 | 7.8918e-06 | 3 | -0.00082543 |
| sanders | dead | 4 | 4 | 7.7614e-06 | 4 | 7.7612e-06 |

The follower and the dead pressure give different buckling pressures, and
the Donnell-type kinematics, accurate for modes with many circumferential
waves, overestimate the follower buckling pressure at $n = 2$, see
[Kinematics of the cylindrical shell](../kinematics.rst).

## References

- **[Hibbitt1979]** Hibbitt, H. D., Some follower forces and load stiffness, International
  Journal for Numerical Methods in Engineering, 14(6), 937-941, 1979,
  doi:10.1002/nme.1620140613.
- **[SchweizerhofRamm1984]** Schweizerhof, K., and Ramm, E., Displacement dependent pressure loads in
  nonlinear finite element analyses, Computers & Structures, 18(6),
  1099-1114, 1984, doi:10.1016/0045-7949(84)90154-8.